# Importing neccessary Libraries

In [1]:
import pandas as pd
import numpy as np
import os
import cv2
import tensorflow as tf
import matplotlib.pyplot as plt
import pickle
import seaborn as sns 

from tensorflow.keras import regularizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
from tensorflow.keras.layers import Dense, Flatten, Dropout,BatchNormalization ,Activation, GaussianNoise, GlobalAveragePooling2D
from tensorflow.keras.models import Model, Sequential
from keras.applications.nasnet import NASNetLarge
from tensorflow.keras.callbacks import ReduceLROnPlateau, ModelCheckpoint, EarlyStopping
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import plot_model
from IPython.display import Image as Img_show
from sklearn.metrics import confusion_matrix

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
data_path =  '/kaggle/input/fer2013/train'
categories = ['angry', 'disgust', 'fear', 'happy', 'Neutral', 'Sad', 'Suprise']

# Reading Image

In [ ]:
for emotion in categories:
  img_fol = os.path.join(data_path, emotion)
  for img_path in os.listdir(img_fol):
      img = cv2.imread(os.path.join(img_fol, img_path))
      img_resize = cv2.resize(img, (100, 100))
      plt.imshow(img_resize)
      plt.title(emotion)
      print(img_path)
      plt.plot()
      break
  break

# Creating DataGens

In [ ]:
Train_Datagen = ImageDataGenerator(
    dtype = 'float32',
    featurewise_center=False,
    featurewise_std_normalization=False,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    preprocessing_function=tf.keras.applications.resnet.preprocess_input,
    )

Test_Datagen = ImageDataGenerator(
    dtype = 'float32',
    preprocessing_function=tf.keras.applications.resnet.preprocess_input,
    )

In [ ]:
train_dataset  = Train_Datagen.flow_from_directory(directory = '/kaggle/input/fer2013/train',
                                                   target_size = (48, 48),
                                                   class_mode = 'categorical',
                                                   batch_size = 64)

In [ ]:
test_dataset = Test_Datagen.flow_from_directory(directory = '/kaggle/input/fer2013/test',
                                                  target_size = (48,48),
                                                  class_mode = 'categorical',
                                                  batch_size = 64)

# Building Model

In [ ]:
# Resnet
base_model = tf.keras.applications.resnet.ResNet101(
    input_shape=(48,48,3),
    include_top=False,
    pooling='max',
    weights="imagenet")

In [ ]:
for layer in base_model.layers[:-50]:
    layer.trainable=False

In [ ]:
# model=Sequential()
# model.add(base_model)

# model.add(Dropout(0.5))
# model.add(Flatten())
# model.add(Dense(128, kernel_initializer='RandomUniform'))
# model.add(BatchNormalization())
# model.add(Activation('relu'))

# model.add(Dropout(0.5))
# model.add(Flatten())
# model.add(Dense(64, kernel_initializer='RandomUniform'))
# model.add(BatchNormalization())
# model.add(Activation('relu'))

# model.add(Dropout(0.5))
# model.add(Flatten())
# model.add(Dense(32, kernel_initializer='RandomUniform'))
# model.add(BatchNormalization())
# model.add(Activation('relu'))

# model.add(Dense(7,activation='softmax'))

In [ ]:
model=Sequential()

model.add(base_model)
model.add(BatchNormalization())
model.add(GaussianNoise(0.01))

model.add(Flatten())
model.add(Dense(256, activation='relu',kernel_regularizer=regularizers.l2(0.001),bias_regularizer=regularizers.l2(0.001)))
model.add(BatchNormalization())
model.add(Dropout(0.5))

model.add(Dense(128, activation='relu',kernel_regularizer=regularizers.l2(0.001),bias_regularizer=regularizers.l2(0.001)))
model.add(BatchNormalization())
model.add(Dropout(0.5))

model.add(Dense(7, activation="softmax"))

In [ ]:
model.summary()

In [ ]:
# plot_model(model, to_file='convnet.png', show_shapes=True,show_layer_names=True)
# Img_show(filename='convnet.png')

In [ ]:
Adam = tf.keras.optimizers.Adam(
    learning_rate=0.0001, 
    beta_1=0.9, 
    beta_2=0.999, 
    epsilon=1e-08)

In [ ]:
early_stopping = EarlyStopping(monitor = 'val_loss', verbose=1, patience=5, restore_best_weights=True)

In [ ]:
model.compile(optimizer='Adam', loss='categorical_crossentropy',metrics=['accuracy'])

# Training Model

In [ ]:
history=model.fit(train_dataset,
                  validation_data=test_dataset,
                  epochs = 60,
                  verbose = 1,
                  callbacks=[early_stopping])

# Visualizing

In [ ]:
train_loss = history.history['loss']
val_loss = history.history['val_loss']
epochs = range(1, len(train_loss) + 1)
plt.plot(epochs, train_loss, 'bo', label='Training loss')
plt.plot(epochs, val_loss, 'b', label='Validation loss')
plt.title('Training and validation loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
train_acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
plt.plot(epochs, train_acc, 'bo', label='Training accuracy')
plt.plot(epochs, val_acc, 'b', label='Validation accuracy')
plt.title('Training and validation accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

In [ ]:
validation_labels = test_dataset.classes
validation_pred_probs = model.predict(test_dataset)
validation_pred_labels = np.argmax(validation_pred_probs, axis=1)

confusion_mtx = confusion_matrix(validation_labels, validation_pred_labels)
class_names = list(train_dataset.class_indices.keys())
sns.set()
plt.figure(figsize=(8, 8))
sns.heatmap(confusion_mtx, annot=True, fmt='d', cmap='YlGnBu', 
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()

# Predicting Image

In [ ]:
img = cv2.imread('/kaggle/input/fer2013/test/happy/PrivateTest_10077120.jpg')
img_resize = cv2.resize(img, (48, 48))
img_processed = np.array(img_resize).reshape(-1, 48, 48,3)

In [ ]:
pred = model.predict(img_processed)
pred_class = categories[np.argmax(pred)]
pred_class

In [ ]:
pred

# Saving model

In [ ]:
model.save('/kaggle/working/model')

In [ ]:
!zip -r file.zip /kaggle/working/model